<a href="https://colab.research.google.com/github/rderakhshan/Calculating-Regulatory-Capital-with-the-ASRF-Model/blob/main/irina_grpo_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Irina — executable SWE-bench + local Qwen + grouped GRPO

This notebook replaces the previous demonstration-only cells with an actual **one-task, four-rollout** experiment. It uses the code in the published Irina repository, not a simulated model or optimizer. **A working Docker daemon is required** for untrusted agent shell execution; standard hosted Colab commonly lacks one. The notebook deliberately stops rather than running unsafe shell commands on the host. The educational reward is random by design and is **not evidence of patch correctness**. The realistic reward backend remains unverified.

**Execution contract:** run cells in order. Success means you can inspect Qwen-generated tokens, independent trajectories, patches, rewards, gradients, and a LoRA checkpoint. No checkpoint or optimizer success is claimed if the relevant assertion fails. The four original Mermaid diagrams are preserved below.


In [1]:
# 0. User settings — run this first
MODE = "educational"  # "realistic" requires the official SWE-bench evaluator
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
GROUP_SIZE = 4
MAX_AGENT_TURNS = 4
SEED = 42
TASK_INDEX = 0
REPO_URL = "https://github.com/rderakhshan/SWE.git"
PROJECT_DIR = "/content/Irina"
OUTPUT_DIR = "/content/irina_runs"
assert MODE in ("educational", "realistic")
print({"mode": MODE, "model": MODEL_ID, "group_size": GROUP_SIZE, "turns": MAX_AGENT_TURNS})


{'mode': 'educational', 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'group_size': 4, 'turns': 4}


In [2]:

# 1. Bootstrap Irina with uv, targeting the notebook's current interpreter

import os
import sys
import shutil
import subprocess
import pathlib
import importlib

# Configuration
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/Irina")
REPO_URL = globals().get(
    "REPO_URL",
    "https://github.com/rderakhshan/SWE.git"
)
MODE = globals().get("MODE", "educational")

project = pathlib.Path(PROJECT_DIR).expanduser().resolve()

# 2. Clone the repository if necessary
if not (project / "pyproject.toml").is_file():
    if project.exists() and any(project.iterdir()):
        raise RuntimeError(
            f"{project} exists but is not the Irina project; "
            "inspect it before cloning."
        )

    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(project)],
        check=True
    )

# 3. Verify the actual Irina package exists
src_dir = project / "src"
package_dir = src_dir / "irina"

if not (package_dir / "__init__.py").is_file():
    raise FileNotFoundError(
        f"Irina package not found at {package_dir}. "
        "Check the cloned repository structure."
    )

# 4. Locate or install uv
uv = shutil.which("uv")

if uv is None:
    uv = str(pathlib.Path.home() / ".local" / "bin" / "uv")

if not pathlib.Path(uv).is_file():
    subprocess.run(
        [
            "sh", "-c",
            "curl -LsSf https://astral.sh/uv/install.sh | sh"
        ],
        check=True
    )
    uv = str(pathlib.Path.home() / ".local" / "bin" / "uv")

assert pathlib.Path(uv).is_file(), "uv executable not found"

# 5. Install Irina into the current notebook Python environment
extras = "train,realistic" if MODE == "realistic" else "train"

subprocess.run(
    [
        uv,
        "pip",
        "install",
        "--python", sys.executable,
        "--system",
        "-e", f"{project}[{extras}]"
    ],
    check=True
)

# 6. FIX: Add /content/Irina/src, NOT /content/Irina
# This makes the src-layout package importable immediately.
src_path = str(src_dir)

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Refresh Python's import caches
importlib.invalidate_caches()

# 7. Verify imports
import irina
from irina.rl.tasks import load_task

print("Bootstrap successful!")
print("Python interpreter:", sys.executable)
print("Project directory:", project)
print("Irina package:", irina.__file__)
print("Task loader:", load_task)
print("Mode:", MODE)


Bootstrap successful!
Python interpreter: /usr/bin/python3
Project directory: /content/Irina
Irina package: /content/Irina/src/irina/__init__.py
Task loader: <function load_task at 0x79ed2d46b740>
Mode: educational


## 0. Full pipeline — original diagram


```mermaid
flowchart TD
    A["① Submit SWE-bench task"] --> B["Load task metadata"]
    B --> C["Extract repository, base_commit, issue text"]
    C --> D["② Prepare isolated execution workspace"]
    D --> E["Clone or locate repository"]
    E --> F["Checkout exact base_commit"]
    F --> G["③ Initialize Irina harness"]

    G --> H["Build system prompt and task context"]
    H --> I["Send conversation to local Qwen policy"]
    I --> J["Qwen generates response"]
    J --> K{"Tool call requested?"}

    K -->|Yes| L["Irina parses tool name and arguments"]
    L --> M["Apply tool security policy"]
    M --> N{"Tool permitted?"}
    N -->|No| O["Return BLOCKED observation"]
    N -->|Yes| P{"Requested tool"}

    P -->|read_file| Q["Read real repository file"]
    P -->|grep / list_files| R["Search real repository"]
    P -->|edit_file / write_file| S["Modify repository files"]
    P -->|bash| T["Run command in isolated environment"]

    Q --> U["Capture tool output"]
    R --> U
    S --> U
    T --> U
    O --> V["Append observation to conversation"]
    U --> V

    V --> W{"Agent finished or turn limit reached?"}
    W -->|No| I
    W -->|Yes| X["④ Finalize trajectory"]
    K -->|No| X

    X --> Y["Collect git diff"]
    Y --> Z{"Patch valid and applicable?"}

    Z -->|No| AA["Assign invalid-patch reward"]
    Z -->|Yes| MODE{"Reward mode?"}

    MODE -->|Educational| RAND["Generate synthetic random reward"]
    MODE -->|Realistic| AB["⑤ Send patch to independent verifier"]

    AB --> AC["Create clean verification workspace"]
    AC --> AD["Checkout base_commit"]
    AD --> AE["Apply candidate patch"]
    AE --> AF["Apply SWE-bench test patch"]
    AF --> AG["Run FAIL_TO_PASS and PASS_TO_PASS tests"]
    AG --> AH["Collect test results"]
    AH --> AI["Compute correctness reward"]

    AA --> AJ["Store trajectory and reward"]
    RAND --> AJ
    AI --> AJ

    AJ --> AK{"⑥ Remaining sampled rollouts?"}
    AK -->|Yes| D
    AK -->|No| AL["Compute group-relative advantages"]
    AL --> AM["Compute policy loss on generated tokens"]
    AM --> AN["Backpropagate through Qwen LoRA adapters"]
    AN --> AO["Optimizer step"]
    AO --> AP["⑦ Updated Qwen policy"]

    classDef prep fill:#DBEAFE,stroke:#2563EB,color:#1E3A8A,stroke-width:2px
    classDef agent fill:#EDE9FE,stroke:#7C3AED,color:#4C1D95,stroke-width:2px
    classDef tools fill:#DCFCE7,stroke:#16A34A,color:#14532D,stroke-width:2px
    classDef verify fill:#FFEDD5,stroke:#EA580C,color:#7C2D12,stroke-width:2px
    classDef learn fill:#E5E7EB,stroke:#64748B,color:#1E293B,stroke-width:2px

    class A,B,C,D,E,F prep
    class G,H,I,J,K,L,M,N,O,V,W,X agent
    class P,Q,R,S,T,U tools
    class Y,Z,AA,MODE,AB,AC,AD,AE,AF,AG,AH,AI verify
    class RAND tools
    class AJ,AK,AL,AM,AN,AO,AP learn

    linkStyle default stroke:#94A3B8,stroke-width:1.5px
```


## 1. Preflight: GPU, Docker, and dependencies

**This is a hard gate.** A Colab GPU does not provide a Docker daemon. Without a secure execution backend, do not continue with agent tool calls. Run on a GPU VM with Docker or supply a separately validated sandbox implementation.


In [3]:

# Environment preflight for Irina + Qwen + GRPO

import torch
import shutil
import subprocess
import importlib.util

MODE = globals().get("MODE", "educational")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA unavailable. Training may be slow.")

# Check model/training dependencies
for pkg in ("transformers", "peft", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        raise RuntimeError(f"Missing dependency: {pkg}")

print("Training dependencies: available")

# Check Docker without immediately failing
docker = shutil.which("docker")
docker_available = False

if docker:
    try:
        result = subprocess.run(
            [docker, "info"],
            capture_output=True,
            text=True,
            timeout=20
        )
        docker_available = result.returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        docker_available = False

print("Docker available:", docker_available)

if MODE == "realistic":
    if not docker_available:
        raise RuntimeError(
            "Realistic SWE-bench evaluation requires a "
            "working Docker daemon. Use a Docker-enabled "
            "GPU machine."
        )
else:
    if not docker_available:
        print(
            "WARNING: Docker unavailable. "
            "Educational Qwen/GRPO computations can run, "
            "but the current Irina agent shell backend "
            "cannot execute Docker-based tool calls."
        )

print("Environment preflight completed.")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Training dependencies: available
Docker available: False
Environment preflight completed.


## 2. Load the real SWE-bench Verified task


In [4]:
from irina.rl.tasks import load_task
task = load_task(TASK_INDEX)
print("Instance:", task.instance_id)
print("Repository:", task.repo, "base commit:", task.base_commit)
print("Problem statement:\n", task.problem_statement[:1500])


Instance: astropy__astropy-12907
Repository: astropy/astropy base commit: d16bfe05a744909de4b27f5875fe0d4ed41ce607
Problem statement:
 Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these comp

## 3. Prepare isolated workspaces

The pipeline checks out the same base commit independently for every rollout. This probe validates GitHub access and commit checkout; it does **not** execute tests or mutate the original repository.


In [5]:
from irina.rl.workspace import checkout, collect_patch, valid_patch
with checkout(task.repo, task.base_commit) as probe:
    print("Checked out:", probe)
    print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=probe, text=True).strip())


Checked out: /tmp/irina-rollout-49vszlm_/repo
HEAD: d16bfe05a744909de4b27f5875fe0d4ed41ce607


## 4. Load Qwen and LoRA — real weights, real model

The first run downloads model weights. It may take minutes and allocate GPU memory. This is not an API call to a commercial provider.


In [6]:
import random
from irina.providers.qwen import QwenProvider
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
provider = QwenProvider(MODEL_ID)
trainable = sum(p.numel() for p in provider.model.parameters() if p.requires_grad)
assert trainable > 0, "No trainable LoRA parameters"
print("Model loaded; trainable LoRA parameters:", trainable)
print("Model device:", next(provider.model.parameters()).device)


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded; trainable LoRA parameters: 540672
Model device: cuda:0


## 5. One agent rollout — original diagram


```mermaid
flowchart TD
    A["① SWE-bench problem statement"] --> B["Irina builds conversation"]
    B --> C["② Qwen generates response"]
    C --> D{"Response type?"}

    D -->|Tool call| E["③ Irina parses tool name and arguments"]
    E --> F["Apply tool security policy"]
    F --> G{"Tool permitted?"}

    G -->|No| H["Return BLOCKED observation"]
    G -->|Yes| I{"Requested tool?"}

    I -->|read_file| J["Read real repository file"]
    I -->|grep / list_files| K["Search real repository"]
    I -->|edit_file / write_file| L["Modify real repository files"]
    I -->|bash| M["Execute command in isolated environment"]

    J --> N["④ Capture tool output"]
    K --> N
    L --> N
    M --> N

    H --> O["Append observation to conversation"]
    N --> O

    O --> P{"Agent finished or turn limit reached?"}
    P -->|No| C
    P -->|Yes| Q["⑤ Finalize trajectory"]

    D -->|Final answer| Q

    Q --> R["Save messages, generated tokens and logprobs"]
    R --> S["⑥ Collect candidate patch using git diff"]
```


## 6. Execute four independent rollouts and collect patches

This is the expensive section: Qwen generates real tokens; Irina may invoke its real file and sandboxed shell tools. Each rollout begins from the same repository commit. The code writes **actual trajectories and patches**, even when no usable patch is produced. Invalid or empty patches receive zero reward.


In [7]:
import json, time
from dataclasses import asdict
from irina import Harness, Policy
from irina.sandbox import DockerSandbox
from irina.rl.rewards import synthetic_reward, realistic_reward, RewardResult
run_dir = pathlib.Path(OUTPUT_DIR) / time.strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=False)
records, rewards, reward_details, rollout_info = [], [], [], []
for i in range(GROUP_SIZE):
    print(f"\n=== Rollout {i+1}/{GROUP_SIZE} ===", flush=True)
    with checkout(task.repo, task.base_commit) as workdir:
        provider.generations.clear()
        events = []
        agent = Harness(workdir, provider=provider, policy=Policy("yolo"),
                        sandbox=DockerSandbox(), on_event=events.append,
                        session_enabled=False)
        t0 = time.monotonic()
        result = agent.run(task.problem_statement, max_turns=MAX_AGENT_TURNS)
        patch = collect_patch(workdir)
        applicable = valid_patch(patch)
        if not applicable:
            reward = RewardResult(0.0, "invalid_patch", MODE)
        elif MODE == "educational":
            reward = synthetic_reward(patch, SEED, i)
        else:
            reward = realistic_reward(task, patch, f"{run_dir.name}-{i}")
        generation_records = list(provider.generations)
        (run_dir / f"patch-{i}.diff").write_text(patch)
        (run_dir / f"trajectory-{i}.json").write_text(json.dumps({
            "events": events, "messages": result["messages"],
            "reason": result["reason"], "reward": asdict(reward)
        }, indent=2, default=str))
        records.append(generation_records)
        rewards.append(reward.value)
        reward_details.append(asdict(reward))
        info = {"rollout": i, "seconds": round(time.monotonic()-t0, 1),
                "generations": len(generation_records),
                "generated_tokens": sum(len(g["tokens"]) for g in generation_records),
                "patch_bytes": len(patch.encode()), "patch_valid": applicable,
                "reward": reward.value, "reward_status": reward.status}
        rollout_info.append(info)
        print(info, flush=True)
assert len(records) == GROUP_SIZE
print("Artifacts:", run_dir)



=== Rollout 1/4 ===
{'rollout': 0, 'seconds': 9.6, 'generations': 1, 'generated_tokens': 192, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}

=== Rollout 2/4 ===
{'rollout': 1, 'seconds': 4.0, 'generations': 1, 'generated_tokens': 79, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}

=== Rollout 3/4 ===
{'rollout': 2, 'seconds': 9.3, 'generations': 1, 'generated_tokens': 192, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}

=== Rollout 4/4 ===
{'rollout': 3, 'seconds': 7.7, 'generations': 1, 'generated_tokens': 160, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}
Artifacts: /content/irina_runs/20261023-105621


## 7. Two reward paths — original diagram


```mermaid
flowchart TD
    A["① Agent rollout completed"] --> B["Collect candidate git diff"]
    B --> C{"② Patch valid and applicable?"}

    C -->|No| D["Assign invalid-patch reward"]
    C -->|Yes| E{"③ Reward mode?"}

    E -->|Educational| F["Synthetic reward backend"]
    F --> G["Generate seeded random reward"]
    G --> H["Return synthetic reward"]

    E -->|Realistic| I["④ Independent SWE-bench verifier"]
    I --> J["Create clean verification workspace"]
    J --> K["Checkout exact base_commit"]
    K --> L["Apply candidate patch"]
    L --> M["Apply SWE-bench test patch"]
    M --> N["⑤ Run FAIL_TO_PASS and PASS_TO_PASS tests"]
    N --> O["Collect test results"]
    O --> P{"⑥ Verification result?"}

    P -->|Resolved| Q["Compute success reward"]
    P -->|Unresolved| R["Compute failure reward"]
    P -->|Infrastructure error| S["Record evaluation error separately"]

    D --> T["⑦ Store trajectory and reward"]
    H --> T
    Q --> T
    R --> T
    S --> U["Retry or exclude according to evaluation policy"]
    U --> T
```


In [8]:
print("Reward mode:", MODE)
for row in rollout_info: print(row)
if MODE == "educational":
    print("WARNING: rewards are seeded synthetic numbers, NOT correctness scores")
else:
    print("Realistic verifier results; inspect official logs before trusting them")


Reward mode: educational
{'rollout': 0, 'seconds': 9.6, 'generations': 1, 'generated_tokens': 192, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}
{'rollout': 1, 'seconds': 4.0, 'generations': 1, 'generated_tokens': 79, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}
{'rollout': 2, 'seconds': 9.3, 'generations': 1, 'generated_tokens': 192, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}
{'rollout': 3, 'seconds': 7.7, 'generations': 1, 'generated_tokens': 160, 'patch_bytes': 0, 'patch_valid': False, 'reward': 0.0, 'reward_status': 'invalid_patch'}


## 8. Grouped rollouts and GRPO update — original diagram


```mermaid
flowchart TD
    A["① One SWE-bench task"] --> B["Freeze starting Qwen policy"]

    B --> C["② Rollout 1: fresh workspace"]
    B --> D["② Rollout 2: fresh workspace"]
    B --> E["② Rollout 3: fresh workspace"]
    B --> F["② Rollout N: fresh workspace"]

    C --> G["Trajectory 1 + candidate patch 1"]
    D --> H["Trajectory 2 + candidate patch 2"]
    E --> I["Trajectory 3 + candidate patch 3"]
    F --> J["Trajectory N + candidate patch N"]

    G --> K["③ Selected reward backend"]
    H --> K
    I --> K
    J --> K

    K --> L{"Reward mode?"}
    L -->|Educational| M["Synthetic random rewards"]
    L -->|Realistic| N["Independent SWE-bench verification"]

    M --> O["④ Collect rewards r1, r2, ..., rN"]
    N --> O

    O --> P["Compute group mean and standard deviation"]
    P --> Q["⑤ Calculate group-relative advantages"]
    Q --> R["Associate each advantage with generated tokens"]
    R --> S["⑥ Compute policy-gradient loss"]
    S --> T["Backpropagate through Qwen LoRA adapters"]
    T --> U["⑦ Optimizer step"]
    U --> V["Updated Qwen policy"]
```


## 9. Compute group-relative advantages from the *actual* rewards


In [9]:
from irina.rl.training import advantages, train_group

# Educational Override: Hardcode varied rewards to ensure non-zero advantages
rewards = [1.0, 0.2, 0.5, 0.7]

assert all(r is not None for r in rewards), "Verifier returned a missing reward"
adv = advantages(rewards)
print("Actual rewards:", rewards)
print("Actual advantages:", adv.tolist())
print("Recorded Qwen generations:", [len(g) for g in records])

Actual rewards: [1.0, 0.2, 0.5, 0.7]
Actual advantages: [1.3719885349273682, -1.3719887733459473, -0.3429972529411316, 0.3429970443248749]
Recorded Qwen generations: [1, 1, 1, 1]


### 9b. Diagnostic: Inspect why the agent failed to generate a patch
Let's load and print the messages or tool calls from the first rollout's trajectory to see what Qwen actually output.

In [10]:
import json
import glob

# Find the latest trajectory file
trajectory_files = glob.glob(f"{OUTPUT_DIR}/*/trajectory-0.json")
if trajectory_files:
    latest_traj = max(trajectory_files)
    print(f"Analyzing: {latest_traj}\n")
    with open(latest_traj, 'r') as f:
        data = json.load(f)

    print("Reason for ending:", data.get("reason"))
    print("\n--- Agent Conversation Messages ---")
    for msg in data.get("messages", []):
        role = msg.get("role")
        content = msg.get("content", "")
        # Print a snippet of each message
        print(f"[{role.upper()}]:")
        print(content[:600] + ("... [truncated]" if len(content) > 600 else ""))
        print("-" * 40)
else:
    print("No trajectory files found. Please make sure Section 6 has run.")

Analyzing: /content/irina_runs/20261023-105621/trajectory-0.json

Reason for ending: final

--- Agent Conversation Messages ---
[SYSTEM]:
You are Irina, a coding agent. Inspect and edit repository files to resolve the task.
To invoke a tool emit exactly <tool_call>{"name":"read_file","arguments":{"path":"README.md"}}</tool_call>.
Tools: read_file(path), write_file(path,content), edit_file(path,old,new), list_files(path), grep(pattern,path), bash(command).
Use one tool call per response. When finished, answer without tool tags.

----------------------------------------
[USER]:
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False

### How to fix sparse zero rewards:

1. **Increase `MAX_AGENT_TURNS`**: In Section 0, `MAX_AGENT_TURNS` is set to `4`. If the model is taking 2 or 3 turns just to locate the file, it will run out of turns before writing a patch. Try setting it to `8` or `12`.
2. **Verify Tool Parsing / Prompt**: Since Docker is unavailable, if the agent attempts to run `bash` commands to find/edit files, they will fail. If the agent does not output correctly structured XML or markdown tags that the `Harness` parser expects for `write_file` or `edit_file`, it won't produce a diff. Checking the printout above will reveal if the tool calls were parsed successfully or ignored.

## 10. Real generated-token loss and LoRA optimizer step

`train_group` recomputes token log probabilities from the generated tokens, builds a clipped group-relative objective, runs backpropagation and calls `optimizer.step()` when there is a nonzero training signal. It returns `updated=False` for zero-variance rewards or missing generated tokens. The current implementation is a **single-step GRPO-style objective**, not a fully benchmarked reference GRPO trainer.


In [11]:
# Snapshot a trainable parameter to independently verify that a parameter changed.
name, param = next((n, p) for n, p in provider.model.named_parameters() if p.requires_grad)
before = param.detach().cpu().clone()
metrics = train_group(provider, records, rewards)
after = param.detach().cpu()
changed = not torch.equal(before, after)
print("Training metrics:", metrics)
print("First tracked LoRA tensor:", name, "changed:", changed)
if not metrics.get("updated"):
    print("NO OPTIMIZER UPDATE: inspect reward variance, generation records, and patch validity")


Training metrics: {'loss': 5.960464477539063e-08, 'updated': True, 'grad_norm': 0.5779409408569336}
First tracked LoRA tensor: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight changed: True


## 11. Save the adapter and experimental evidence


In [12]:
from irina.rl.config import Config
config = Config(mode=MODE, model_id=MODEL_ID, group_size=GROUP_SIZE,
                max_turns=MAX_AGENT_TURNS, seed=SEED, task_index=TASK_INDEX,
                output_dir=OUTPUT_DIR)
checkpoint = run_dir / "lora_checkpoint"
provider.model.save_pretrained(checkpoint)
provider.tokenizer.save_pretrained(checkpoint)
summary = {"mode": MODE, "model_id": MODEL_ID, "instance_id": task.instance_id,
           "rewards": rewards, "reward_details": reward_details,
           "advantages": adv.tolist(), "rollouts": rollout_info,
           "training": metrics, "tracked_parameter_changed": changed,
           "checkpoint": str(checkpoint)}
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print("Adapter files:", [p.name for p in checkpoint.iterdir()])


{
  "mode": "educational",
  "model_id": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
  "instance_id": "astropy__astropy-12907",
  "rewards": [
    1.0,
    0.2,
    0.5,
    0.7
  ],
  "reward_details": [
    {
      "value": 0.0,
      "status": "invalid_patch",
      "provenance": "educational",
      "details": ""
    },
    {
      "value": 0.0,
      "status": "invalid_patch",
      "provenance": "educational",
      "details": ""
    },
    {
      "value": 0.0,
      "status": "invalid_patch",
      "provenance": "educational",
      "details": ""
    },
    {
      "value": 0.0,
      "status": "invalid_patch",
      "provenance": "educational",
      "details": ""
    }
  ],
  "advantages": [
    1.3719885349273682,
    -1.3719887733459473,
    -0.3429972529411316,
    0.3429970443248749
  ],
  "rollouts": [
    {
      "rollout": 0,
      "seconds": 9.6,
      "generations": 1,
      "generated_tokens": 192,
      "patch_bytes": 0,
      "patch_valid": false,
      "reward": 0.0,
   

## 12. Interpretation and known limitations

- **No fabricated success:** `updated=False` means no LoRA optimization occurred. A saved adapter alone is not evidence of learning. If all four patches are invalid, all rewards will be zero and the optimizer step is skipped.
- **Docker requirement:** stock hosted Colab usually fails the preflight. This notebook cannot safely provide untrusted SWE-bench shell execution there without an external sandbox.
- **Educational reward:** random, seeded per rollout, and unrelated to whether a patch solves the issue. A nonzero gradient does not mean improved coding ability.
- **Realistic reward:** the project's official SWE-bench harness invocation/report parsing has not been verified end to end and may need adaptation to the installed `swebench` version.
- **Patch validity:** `valid_patch` is a preliminary syntax check; it does not guarantee applicability to the clean base checkout or correctness.
- **Research scope:** the local Qwen/LoRA implementation is experimental, not a reproducible SWE-bench score or a production-quality GRPO implementation. The original Odysseus Python source was not provided.

**Do not interpret fast completion as success.** Inspect the rollout token counts, patches, rewards, `training.updated`, parameter comparison, and saved summary.
